<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
ChromaDB &amp; Indexing
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M12-ChromaDB-Indexing"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

In [ ]:
#@title 📂 Dokumente, Bilder { display-mode: "form" }
!rm -rf /content/files
!mkdir /content/files
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_1.txt  -o /content/files/biografien_1.txt
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_2.md   -o /content/files/biografien_2.md
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_3.pdf  -o /content/files/biografien_3.pdf
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_4.docx -o /content/files/biografien_4.docx

# 1 | Übersicht
---


ChromaDB läuft lokal, braucht keine externe Infrastruktur und lässt sich in wenigen Zeilen mit LangChain verbinden — das macht es zur sinnvollen ersten Wahl für RAG-Prototypen.

## Warum ChromaDB?

| Feature | ChromaDB | Alternativen |
|---------|---------|-------------|
| Installation | `pip install chromadb` | Faiss: C-Build nötig |
| Persistenz | Dateisystem (SQLite) | Pinecone: Cloud-only |
| Metadaten-Filter | ✅ Eingebaut | Weaviate: komplexer |
| LangChain-Integration | ✅ Nahtlos | Alle unterstützt |
| Lokaler Betrieb | ✅ Vollständig | Pinecone: nur Cloud |

## In diesem Modul

1. ChromaDB einrichten (in-memory & persistent)
2. Dokumente indexieren (laden, chunken, embedden)
3. Collections verwalten (erstellen, filtern, aktualisieren)
4. Häufige Fehler und Lösungen

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    subgraph Indexierung
        DOCS["📄 Dokumente"] --> LOADER["📥 Document Loader"]
        LOADER --> SPLITTER["✂️ Text Splitter"]
        SPLITTER --> EMBED["🔢 OpenAI Embeddings"]
        EMBED --> CHROMA[("🗄️ ChromaDB")]
    end
    subgraph Suche
        Q["❓ Query"] --> QE["🔢 Query Embed"]
        QE --> SIM["🔍 Similarity Search"]
        CHROMA --> SIM
        SIM --> RESULT["📋 Top-K Docs"]
    end
'''

mermaid(diagram, width=1100)

# 2 | ChromaDB Setup
---


ChromaDB kann auf zwei Arten betrieben werden: Im **In-Memory-Modus** existieren die Daten nur im RAM — geeignet für schnelle Demos und Tests, aber nach dem Prozessende verloren. Im **persistenten Modus** werden die Daten auf dem Dateisystem gespeichert: lokal direkt im Projektverzeichnis, in Google Colab über ein eingebundenes Google Drive, da das Colab-Dateisystem temporär ist und alle Daten beim Neustart verloren gehen.

**Wichtig für Google Colab:**     
Das Colab-Dateisystem ist temporär — alle Daten gehen beim Neustart verloren.  
Für echte Persistenz muss Google Drive eingebunden werden.


In [ ]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

CHROMA_DIR = "/content/chroma_m12"

# Embedding-Modell
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ChromaDB persistent initialisieren
vectorstore = Chroma(
    collection_name="agenten_docs",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

print(f"\nChromaDB initialisiert")
print(f"  Speicherort: {CHROMA_DIR}")
print(f"  Collection:  agenten_docs")
print(f"  Dokumente:   {vectorstore._collection.count()}")

# 3 | Dokumente indexieren
---



Das Indexieren umfasst drei Schritte:

1. **Laden** – Dokumente einlesen (Text, PDF, Markdown, ...)
2. **Chunken** – In kleinere Abschnitte aufteilen
3. **Einbetten** – Vektoren erstellen und in ChromaDB speichern


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_01.png" width="650" alt="Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB">

*Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB*


<p><font color='black' size="5">
Dokument-Loader in LangChain
</font></p>


| Loader | Dateityp | Import |
|--------|---------|--------|
| `TextLoader` | .txt | `langchain_community.document_loaders` |
| `PyPDFLoader` | .pdf | `langchain_community.document_loaders` |
| `UnstructuredMarkdownLoader` | .md | `langchain_community.document_loaders` |
| `DirectoryLoader` | Verzeichnis | `langchain_community.document_loaders` |

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*triton.*")

from langchain_community.document_loaders import (
    UnstructuredMarkdownLoader,
    UnstructuredWordDocumentLoader,
    PyPDFLoader,
    UnstructuredFileLoader,
    DirectoryLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader-Konfiguration (Dateityp → passender Loader)
loader_mapping = {
    "*.md":   UnstructuredMarkdownLoader,
    "*.docx": UnstructuredWordDocumentLoader,
    "*.pdf":  PyPDFLoader,
    "*.txt":  UnstructuredFileLoader,
}

def load_documents_from_directory(directory_path):
    """Lädt Dokumente aus dem Verzeichnis für alle unterstützten Dateitypen."""
    documents = []
    for file_pattern, loader_cls in loader_mapping.items():
        loader = DirectoryLoader(directory_path, glob=file_pattern, loader_cls=loader_cls)
        documents.extend(loader.load())
    return documents

# Dokumente laden
dokumente_raw = load_documents_from_directory("/content/files")

# Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)
chunks = splitter.split_documents(dokumente_raw)

print(f"Geladene Dokumente: {len(dokumente_raw)}")
print(f"Chunks nach Splitting: {len(chunks)}")
for i, chunk in enumerate(chunks[:5]):
    quelle = chunk.metadata.get("source", "?").split("/")[-1]
    print(f"  Chunk {i+1}: {len(chunk.page_content)} Zeichen | Quelle: {quelle}")
if len(chunks) > 5:
    print(f"  ... und {len(chunks) - 5} weitere Chunks")

In [ ]:
# Chunks in ChromaDB indexieren — nur wenn Collection noch leer ist
count_vorher = vectorstore._collection.count()
if count_vorher == 0:
    vectorstore.add_documents(chunks)
    print(f"Indexiert: {vectorstore._collection.count()} Dokumente neu angelegt")
else:
    print(f"Collection hat bereits {count_vorher} Eintraege — kein erneutes Indexieren")

# 4 | Collections verwalten
---



Eine **Collection** in ChromaDB ist eine benannte Gruppe von Dokumenten – vergleichbar mit einer Tabelle in SQL.



<p><font color='black' size="5">
Wichtige Operationen
</font></p>


| Operation | Methode | Beschreibung |
|-----------|---------|-------------|
| Dokument hinzufügen | `add_documents()` | Neue Chunks indexieren |
| Ähnlichkeitssuche | `similarity_search()` | Top-K ähnliche Dokumente |
| Mit Score | `similarity_search_with_score()` | Suche + Ähnlichkeitswert |
| Metadaten-Filter | `filter={...}` | Nur bestimmte Dokumente |
| Alle Dokumente | `get()` | Rohe Collection-Daten |

In [ ]:
# Einfache Aehnlichkeitssuche mit Query-Rewrite aus Prompt-Template
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm_rewrite = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
rewrite_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m12_query_rewrite_prompt.md", mode="T")

frage_raw = "Welche Methode hat Nadia Khoury entwickelt, um Mondstaub zu nutzen, und für welches Projekt arbeitet sie?"
frage = (rewrite_prompt | llm_rewrite | StrOutputParser()).invoke({"user_question": frage_raw}).strip()

ergebnisse = vectorstore.similarity_search(frage, k=2)

print(f"Originalfrage: {frage_raw}")
print(f"Retrieval-Query: {frage}")
print()
print("Top-2 Ergebnisse:")
for i, doc in enumerate(ergebnisse):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"{i+1}. [{quelle}] {doc.page_content[:120]}...")

In [ ]:
# Suche mit Ähnlichkeitsscore
ergebnisse_mit_score = vectorstore.similarity_search_with_score(
    "Was hat Dorian Blackwood vor seiner Karriere bei IllusionSec gemacht, und wie hilft er heute Kindern?",
    k=3
)

mprint("## Suchergebnis mit Scores\n")
rows = []
for doc, score in ergebnisse_mit_score:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    rows.append(f"| {quelle} | {score:.4f} | {doc.page_content[:80]}... |")

mprint("| Quelle | Score | Inhalt (Vorschau) |\n|--------|-------|-----------------|\n" + "\n".join(rows))

In [ ]:
# Metadaten-Filter: Nur Dokumente aus einer bestimmten Quelldatei suchen
gefiltert = vectorstore.similarity_search(
    "Welche zwei Fachrichtungen verbindet Raoul Mendez in seiner Arbeit, und was hat sein Projekt EarthEars bereits entdeckt?",
    k=2,
    filter={"source": "/content/files/biografien_2.md"}  # Nur aus biografien_2.md
)

print("Gefilterte Suche (nur biografien_2.md):")
for doc in gefiltert:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"  - {quelle}: {doc.page_content[:100]}...")

# 5 | ChromaDB Troubleshooting
---

**Häufige Fehler und Lösungen**

| Problem | Symptom | Lösung |
|---------|---------|--------|
| **Doppelte Indexierung** | Dokumente mehrfach vorhanden | IDs prüfen, Collection neu aufbauen |
| **Falsches persist_directory** | Daten weg nach Neustart | Absoluten Pfad verwenden |
| **Falsche Collection** | Leere Suchergebnisse | `collection_name` prüfen |
| **Embedding-Modell-Mismatch** | Dimension Error | Immer gleiches Modell für Index & Query |
| **Zu wenige Chunks** | k > Anzahl Dokumente | k ≤ Anzahl indexierter Chunks setzen |


**Best Practices**

```python
# IMMER: Gleiche Embeddings für Index und Query
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # Einmal definieren

# IMMER: Umgebungsabhängigen Pfad verwenden
import os
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CHROMA_DIR = "/content/drive/MyDrive/chroma_m12"   # Colab: Google Drive
except ImportError:
    CHROMA_DIR = os.path.abspath("../02_daten/05_sonstiges/chroma_m12")  # Lokal

# TIPP: Nicht erneut indexieren, wenn Collection bereits Daten hat
count = vectorstore._collection.count()
if count == 0:
    vectorstore.add_documents(chunks)
    print("Neu indexiert.")
else:
    print(f"Collection hat bereits {count} Eintraege — kein erneutes Indexieren nötig.")
```



In [ ]:
# Diagnose: Collection-Status prüfen
count = vectorstore._collection.count()
print(f"Dokumente in Collection: {count}")

# Vorhandene Quelldateien anzeigen
raw = vectorstore._collection.get(include=["metadatas"])
quellen = set(m.get("source", "?").split("/")[-1] for m in raw["metadatas"])
print(f"Vorhandene Quelldateien: {quellen}")

# Tipp: Collection leeren und neu aufbauen
# vectorstore._collection.delete(where={"source": {"$contains": "biografien_1"}})
print("\nCollection-Diagnose abgeschlossen.")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M12-ChromaDB-Indexing", limit=3, show_steps=True)

# A | Aufgabe
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Wissensdatenbank für KI-Agenten
</font></p>

Bauen Sie eine persistente Wissensdatenbank für das Agenten-Kurs-Material auf.

**Grundlagen**

1. Erstellen Sie eine ChromaDB-Collection (persistent oder in-memory).
2. Laden Sie **genau 2 Textdokumente** und indexieren Sie diese in der Collection.
3. Testen Sie **eine** Suchanfrage und geben Sie das Ergebnis aus.

**Aufbau**

1. Laden Sie **mindestens 3 Textdokumente** aus `../02_daten/01_text/` (biografien_*.txt, *.md).
2. Konfigurieren Sie den `RecursiveCharacterTextSplitter` explizit (`chunk_size=400`, `chunk_overlap=40`).
3. Indexieren Sie die Chunks in einer persistenten ChromaDB-Collection.
4. Führen Sie **3 verschiedene Suchanfragen** mit unterschiedlichen `k`-Werten durch.
5. Zeigen Sie mindestens eine Suche mit **Metadaten-Filter** (z. B. `filter={"source": ...}`).
6. Stellen Sie die Ergebnisse als Markdown-Tabelle dar.

**Vertiefung**

- Nutzen Sie Metadaten-Filter, um die Suche auf eine bestimmte Dateiquelle einzuschränken.
- Aktivieren Sie LangSmith-Tracing für die Suchanfragen und analysieren Sie den Trace.

In [ ]:
# ✅ Selbstcheck Grundlagen 🟢
# Führen Sie diesen Block aus, nachdem Sie Ihre Collection mit 2 Dokumenten befüllt haben.
# Passen Sie den Variablennamen ggf. an Ihre Implementierung an (z. B. meine_collection statt collection).

assert collection is not None, "collection ist None – bitte die ChromaDB-Collection erstellen."
assert collection.count() >= 2, f"Nur {collection.count()} Dokument(e) – mindestens 2 indexieren."
print("✅ Grundlagen-Selbstcheck bestanden!")

In [ ]:
# ✅ Selbstcheck Aufbau 🟡
# Führen Sie diesen Block aus, nachdem Sie 3+ Dokumente indexiert und 3 Suchanfragen durchgeführt haben.
# Passen Sie collection und results an Ihre Variablennamen an.

assert collection is not None, "collection ist None – bitte die ChromaDB-Collection erstellen."
assert collection.count() >= 3, f"Nur {collection.count()} Dokument(e) – mindestens 3 indexieren."
assert len(results) > 0, "results ist leer – bitte eine Suchanfrage ausführen und Ergebnis in 'results' speichern."
print("✅ Aufbau-Selbstcheck bestanden!")

In [ ]:
# ✅ Selbstcheck Vertiefung 🔴
import os

assert os.environ.get("LANGSMITH_TRACING") == "true", "LANGSMITH_TRACING ist nicht aktiv – Umgebungsvariable prüfen."
print("✅ Vertiefung-Selbstcheck bestanden!")